<a href="https://colab.research.google.com/github/gaurkumarsoni/Mental-Health-Detection-and-Monitoring-Using-XAI-and-Multimodality/blob/main/Text_Modality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import subprocess
subprocess.run(['pip', 'install', '--force-reinstall', '-q', 'numpy==1.26.4'],
               capture_output=True)
print("✅ Numpy reinstalled")

✅ Numpy reinstalled


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np

BASE_DIR = '/content/drive/MyDrive/depression-ai'
DEVICE   = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
print(f"✅ Drive mounted | Device: {DEVICE}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted | Device: cuda


In [2]:
!pip install -q captum
!pip install -q accelerate
print("✅ Done")

✅ Done


**Loading tweet data**

In [3]:
tweet_path = f'{BASE_DIR}/data/raw/tweets'
df_tweets  = pd.read_csv(f'{tweet_path}/Mental-Health-Twitter.csv')

print(f"Shape    : {df_tweets.shape}")
print(f"Columns  : {df_tweets.columns.tolist()}")
print(f"\nLabel distribution:\n{df_tweets.iloc[:, -1].value_counts()}")
df_tweets.head(3)

Shape    : (20000, 11)
Columns  : ['Unnamed: 0', 'post_id', 'post_created', 'post_text', 'user_id', 'followers', 'friends', 'favourites', 'statuses', 'retweets', 'label']

Label distribution:
label
1    10000
0    10000
Name: count, dtype: int64


,Unnamed: 0,post_id,post_created,post_text,user_id,followers,friends,favourites,statuses,retweets,label
0,0,637894677824413696,Sun Aug 30 07:48:37 +0000 2015,It's just over 2 years since I was diagnosed w...,1013187241,84,211,251,837,0,1
1,1,637890384576778240,Sun Aug 30 07:31:33 +0000 2015,"It's Sunday, I need a break, so I'm planning t...",1013187241,84,211,251,837,1,1
2,2,637749345908051968,Sat Aug 29 22:11:07 +0000 2015,Awake but tired. I need to sleep but my brain ...,1013187241,84,211,251,837,0,1


In [4]:
# Find transcript files
transcript_path = f'{BASE_DIR}/data/raw/daic-woz'
print("DAIC-WoZ contents:")
for f in sorted(os.listdir(transcript_path)):
    print(f"  {f}")

DAIC-WoZ contents:
  Transcripts
  dev_split_Depression_AVEC2017.csv
  full_test_split.csv
  train_split_Depression_AVEC2017.csv


In [5]:
# Check inside Transcripts folder
transcript_path = f'{BASE_DIR}/data/raw/daic-woz/Transcripts'
files = os.listdir(transcript_path)
print(f"Total transcript files: {len(files)}")
print(f"\nFirst 10 files: {sorted(files)[:10]}")

# Load one file to see structure
sample_file = sorted(files)[0]
print(f"\nSample file: {sample_file}")

# Check if csv or txt
if sample_file.endswith('.csv'):
    df = pd.read_csv(f'{transcript_path}/{sample_file}')
elif sample_file.endswith('.txt'):
    with open(f'{transcript_path}/{sample_file}', 'r') as f:
        print(f.read()[:500])
    df = None

if df is not None:
    print(f"\nShape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"\nFirst 3 rows:")
    print(df.head(3))

Total transcript files: 189

First 10 files: ['300_TRANSCRIPT.csv', '301_TRANSCRIPT.csv', '302_TRANSCRIPT.csv', '303_TRANSCRIPT.csv', '304_TRANSCRIPT.csv', '305_TRANSCRIPT.csv', '306_TRANSCRIPT.csv', '307_TRANSCRIPT.csv', '308_TRANSCRIPT.csv', '309_TRANSCRIPT.csv']

Sample file: 300_TRANSCRIPT.csv

Shape: (174, 1)
Columns: ['start_time\tstop_time\tspeaker\tvalue']

First 3 rows:
               start_time\tstop_time\tspeaker\tvalue
0  36.588\t39.668\tEllie\thi i'm ellie thanks for...
1  39.888\t43.378\tEllie\ti was created to talk t...
2  43.728\t48.498\tEllie\tthink of me as a friend...


# **Complete Text Modality**

### **Load & Fix Transcripts**

In [6]:
def load_transcript(participant_id, transcript_path):
    """
    Load transcript for one participant.
    Returns only the PARTICIPANT (not Ellie) speech combined as one string.
    """
    file_path = f'{transcript_path}/{participant_id}_TRANSCRIPT.csv'
    if not os.path.exists(file_path):
        return None

    # Fix: read as tab-separated
    df = pd.read_csv(file_path, sep='\t')

    # Keep only participant speech (not Ellie the interviewer)
    participant_speech = df[df['speaker'] == 'Participant']['value'].dropna()

    if len(participant_speech) == 0:
        return None

    # Combine all utterances into one text
    text = ' '.join(participant_speech.astype(str).tolist())
    return text.strip()


# Test with one participant
transcript_path = f'{BASE_DIR}/data/raw/daic-woz/Transcripts'
sample_text = load_transcript(300, transcript_path)
print(f"Sample transcript (first 300 chars):")
print(sample_text[:300])
print(f"\nTotal words: {len(sample_text.split())}")

Sample transcript (first 300 chars):
good atlanta georgia um my parents are from here um i love it i like the weather i like the opportunities um yes um it took a minute somewhat easy congestion that's it um i took up business and administration uh yeah i am here and there i'm on a break right now but i plan on going back in the uh nex

Total words: 352


**Build DAIC-WoZ Text Dataset**

In [7]:
# Load labels
train_df = pd.read_csv(f'{BASE_DIR}/data/raw/daic-woz/train_split_Depression_AVEC2017.csv')
dev_df   = pd.read_csv(f'{BASE_DIR}/data/raw/daic-woz/dev_split_Depression_AVEC2017.csv')

def build_transcript_dataset(labels_df, transcript_path):
    texts, labels, ids = [], [], []

    for _, row in labels_df.iterrows():
        pid   = int(row['Participant_ID'])
        label = int(row['PHQ8_Binary'])
        text  = load_transcript(pid, transcript_path)

        if text is not None:
            texts.append(text)
            labels.append(label)
            ids.append(pid)

    return texts, labels, ids


print("Loading DAIC-WoZ transcripts...")
train_texts, train_labels, train_ids = build_transcript_dataset(train_df, transcript_path)
dev_texts,   dev_labels,   dev_ids   = build_transcript_dataset(dev_df,   transcript_path)

print(f"Train: {len(train_texts)} | Depressed: {sum(train_labels)} | Not: {len(train_labels)-sum(train_labels)}")
print(f"Dev  : {len(dev_texts)}   | Depressed: {sum(dev_labels)}   | Not: {len(dev_labels)-sum(dev_labels)}")
print(f"\nSample train text (100 chars): {train_texts[0][:100]}")

Loading DAIC-WoZ transcripts...
Train: 107 | Depressed: 30 | Not: 77
Dev  : 35   | Depressed: 12   | Not: 23

Sample train text (100 chars): okay how 'bout yourself here in california yeah oh well that it's big and broad there's a lot to do 


**Prepare Tweet Dataset**

In [12]:
df_tweets = pd.read_csv(f'{BASE_DIR}/data/raw/tweets/Mental-Health-Twitter.csv')
df_tweets = df_tweets[['post_text', 'label']].dropna()

import re

def clean_tweet(text):
    text = str(text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_tweets['post_text'] = df_tweets['post_text'].apply(clean_tweet)
df_tweets = df_tweets[df_tweets['post_text'].str.len() > 10]

# Fixed sampling — no groupby needed
df_dep     = df_tweets[df_tweets['label'] == 1].sample(1500, random_state=42)
df_not_dep = df_tweets[df_tweets['label'] == 0].sample(1500, random_state=42)
df_sample  = pd.concat([df_dep, df_not_dep]).reset_index(drop=True)

tweet_texts  = df_sample['post_text'].tolist()
tweet_labels = df_sample['label'].tolist()

print(f"Tweet samples: {len(tweet_texts)}")
print(f"Label dist: {pd.Series(tweet_labels).value_counts().to_dict()}")
print(f"\nSample tweet: {tweet_texts[0][:100]}")

Tweet samples: 3000
Label dist: {1: 1500, 0: 1500}

Sample tweet: RT : "Bought a Canadian flag for the sole purpose of hanging it below the American flag. MURICA " - 


**Combine DAIC-WoZ + Tweets**

In [13]:
# Combine both datasets for training
all_train_texts  = train_texts + tweet_texts
all_train_labels = train_labels + tweet_labels

print(f"Combined training size : {len(all_train_texts)}")
print(f"Depressed              : {sum(all_train_labels)}")
print(f"Not depressed          : {len(all_train_labels) - sum(all_train_labels)}")
print(f"\nValidation size (DAIC only): {len(dev_texts)}")

# Shuffle combined training data
import random
random.seed(42)
combined = list(zip(all_train_texts, all_train_labels))
random.shuffle(combined)
all_train_texts, all_train_labels = zip(*combined)
all_train_texts  = list(all_train_texts)
all_train_labels = list(all_train_labels)

print(f"\n✅ Data combined and shuffled")

Combined training size : 3107
Depressed              : 1530
Not depressed          : 1577

Validation size (DAIC only): 35

✅ Data combined and shuffled


In [14]:
import re

# ── 1. Check for missing/empty texts ──
print("=== PRE-PROCESSING CHECKS ===\n")

def check_dataset(texts, labels, name):
    total    = len(texts)
    empty    = sum(1 for t in texts if not t or len(t.strip()) == 0)
    too_short = sum(1 for t in texts if len(t.split()) < 5)
    nulls    = sum(1 for t in texts if t is None)

    print(f"[{name}]")
    print(f"  Total samples  : {total}")
    print(f"  Null texts     : {nulls}")
    print(f"  Empty texts    : {empty}")
    print(f"  Too short (<5 words): {too_short}")
    print(f"  Label dist     : {pd.Series(labels).value_counts().to_dict()}")
    print()

check_dataset(all_train_texts, all_train_labels, "Combined Train")
check_dataset(dev_texts,       dev_labels,        "DAIC-WoZ Dev")

# ── 2. Clean all texts ──
def clean_text(text):
    if not text or pd.isna(text):
        return None
    text = str(text).lower()                        # lowercase
    text = re.sub(r'http\S+', '', text)             # remove URLs
    text = re.sub(r'@\w+', '', text)               # remove mentions
    text = re.sub(r'#(\w+)', r'\1', text)          # remove # keep word
    text = re.sub(r'\d+', '', text)                # remove numbers
    text = re.sub(r'[^\w\s\'\-]', ' ', text)      # remove special chars
    text = re.sub(r'\s+', ' ', text).strip()       # normalize whitespace
    return text if len(text.split()) >= 5 else None # min 5 words

# Apply cleaning
print("Cleaning texts...")
cleaned_train = [(clean_text(t), l)
                 for t, l in zip(all_train_texts, all_train_labels)]
cleaned_dev   = [(clean_text(t), l)
                 for t, l in zip(dev_texts, dev_labels)]

# Remove None entries
cleaned_train = [(t, l) for t, l in cleaned_train if t is not None]
cleaned_dev   = [(t, l) for t, l in cleaned_dev   if t is not None]

all_train_texts,  all_train_labels = zip(*cleaned_train)
dev_texts,        dev_labels       = zip(*cleaned_dev)

all_train_texts  = list(all_train_texts)
all_train_labels = list(all_train_labels)
dev_texts        = list(dev_texts)
dev_labels       = list(dev_labels)

# ── 3. Check for duplicates ──
train_unique = len(set(all_train_texts))
print(f"Duplicate texts removed: {len(all_train_texts) - train_unique}")

# Remove duplicates while keeping labels
seen   = set()
deduped_texts, deduped_labels = [], []
for t, l in zip(all_train_texts, all_train_labels):
    if t not in seen:
        seen.add(t)
        deduped_texts.append(t)
        deduped_labels.append(l)

all_train_texts  = deduped_texts
all_train_labels = deduped_labels

# ── 4. Check text length distribution ──
train_lengths = [len(t.split()) for t in all_train_texts]
dev_lengths   = [len(t.split()) for t in dev_texts]

print(f"\n=== TEXT LENGTH STATS ===")
print(f"Train — Min: {min(train_lengths)} | "
      f"Max: {max(train_lengths)} | "
      f"Mean: {np.mean(train_lengths):.0f} | "
      f"Median: {np.median(train_lengths):.0f}")
print(f"Dev   — Min: {min(dev_lengths)} | "
      f"Max: {max(dev_lengths)} | "
      f"Mean: {np.mean(dev_lengths):.0f} | "
      f"Median: {np.median(dev_lengths):.0f}")

# ── 5. Final dataset summary ──
print(f"\n=== FINAL CLEAN DATASET ===")
print(f"Train size : {len(all_train_texts)}")
print(f"Dev size   : {len(dev_texts)}")
print(f"Train label dist: {pd.Series(all_train_labels).value_counts().to_dict()}")
print(f"Dev label dist  : {pd.Series(dev_labels).value_counts().to_dict()}")

# ── 6. Sample check ──
print(f"\nSample cleaned train text:")
print(f"  '{all_train_texts[0][:150]}'")
print(f"\nSample cleaned dev text:")
print(f"  '{dev_texts[0][:150]}'")
print("\n✅ Preprocessing complete!")

=== PRE-PROCESSING CHECKS ===

[Combined Train]
  Total samples  : 3107
  Null texts     : 0
  Empty texts    : 0
  Too short (<5 words): 447
  Label dist     : {0: 1577, 1: 1530}

[DAIC-WoZ Dev]
  Total samples  : 35
  Null texts     : 0
  Empty texts    : 0
  Too short (<5 words): 0
  Label dist     : {0: 23, 1: 12}

Cleaning texts...
Duplicate texts removed: 35

=== TEXT LENGTH STATS ===
Train — Min: 5 | Max: 4112 | Mean: 70 | Median: 13
Dev   — Min: 459 | Max: 3036 | Mean: 1520 | Median: 1401

=== FINAL CLEAN DATASET ===
Train size : 2578
Dev size   : 35
Train label dist: {1: 1306, 0: 1272}
Dev label dist  : {0: 23, 1: 12}

Sample cleaned train text:
  'do not break the law'

Sample cleaned dev text:
  'i'm fine how about yourself i'm from los angeles california what part okay um all my family's here friends a mixture of people and a lot of things to '

✅ Preprocessing complete!


In [16]:
import torch

# Update class weights to reflect new distribution
class_counts  = np.bincount(list(all_train_labels))
class_weights = torch.tensor(
    [1.0/class_counts[0], 1.0/class_counts[1]],
    dtype=torch.float32
).to(DEVICE)
print(f"Updated class weights: {class_weights}")


# Upsample DAIC-WoZ training transcripts 5x
# so model learns clinical language better
daic_texts  = train_texts[:107]   # original DAIC train texts
daic_labels = train_labels[:107]  # original DAIC train labels

# Verify they loaded correctly
print(f"DAIC train texts : {len(daic_texts)}")
print(f"Sample DAIC text : {daic_texts[0][:100]}")

# Upsample DAIC 5x
upsampled_texts  = daic_texts  * 5
upsampled_labels = daic_labels * 5

# Add to training set
all_train_texts  = all_train_texts  + upsampled_texts
all_train_labels = all_train_labels + upsampled_labels

# Shuffle again
combined = list(zip(all_train_texts, all_train_labels))
random.shuffle(combined)
all_train_texts, all_train_labels = zip(*combined)
all_train_texts  = list(all_train_texts)
all_train_labels = list(all_train_labels)

print(f"\n=== FINAL TRAINING SET ===")
print(f"Total samples : {len(all_train_texts)}")
print(f"Label dist    : {pd.Series(all_train_labels).value_counts().to_dict()}")
print(f"\n✅ DAIC transcripts upsampled 5x — clinical language weighted higher")

Updated class weights: tensor([0.0006, 0.0007], device='cuda:0')
DAIC train texts : 107
Sample DAIC text : okay how 'bout yourself here in california yeah oh well that it's big and broad there's a lot to do 

=== FINAL TRAINING SET ===
Total samples : 3648
Label dist    : {0: 2042, 1: 1606}

✅ DAIC transcripts upsampled 5x — clinical language weighted higher


In [17]:
# Reset to clean combined dataset from scratch
import random

# Reload clean combined data
all_train_texts  = deduped_texts
all_train_labels = deduped_labels

# Upsample DAIC 5x only
daic_texts_up   = train_texts * 5
daic_labels_up  = train_labels * 5

all_train_texts  = all_train_texts  + daic_texts_up
all_train_labels = all_train_labels + daic_labels_up

# Shuffle
combined = list(zip(all_train_texts, all_train_labels))
random.shuffle(combined)
all_train_texts, all_train_labels = zip(*combined)
all_train_texts  = list(all_train_texts)
all_train_labels = list(all_train_labels)

# Update class weights
class_counts  = np.bincount(list(all_train_labels))
class_weights = torch.tensor(
    [1.0/class_counts[0], 1.0/class_counts[1]],
    dtype=torch.float32
).to(DEVICE)

print(f"Total samples : {len(all_train_texts)}")
print(f"Label dist    : {pd.Series(all_train_labels).value_counts().to_dict()}")
print(f"Class weights : {class_weights}")
print("✅ Reset complete")

Total samples : 3113
Label dist    : {0: 1657, 1: 1456}
Class weights : tensor([0.0006, 0.0007], device='cuda:0')
✅ Reset complete


**Tokenize With BERT Tokenizer**

In [21]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

MAX_LEN    = 256
BATCH_SIZE = 16

def tokenize_texts(texts, max_len=MAX_LEN):
    return tokenizer(
        texts,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

print("Tokenizing training data...")
train_encodings = tokenize_texts(all_train_texts)

print("Tokenizing validation data...")
dev_encodings   = tokenize_texts(dev_texts)

print(f"✅ Tokenization complete")
print(f"Input IDs shape     : {train_encodings['input_ids'].shape}")
print(f"Attention mask shape: {train_encodings['attention_mask'].shape}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Tokenizing training data...
Tokenizing validation data...
✅ Tokenization complete
Input IDs shape     : torch.Size([3113, 256])
Attention mask shape: torch.Size([3113, 256])


**PyTorch Dataset & DataLoader**

In [22]:
import torch
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids'     : self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'label'         : self.labels[idx]
        }


train_dataset = TextDataset(train_encodings, list(all_train_labels))
dev_dataset   = TextDataset(dev_encodings,   dev_labels)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
dev_loader    = DataLoader(dev_dataset,   batch_size=BATCH_SIZE, shuffle=False)

# Class weights
import numpy as np
class_counts  = np.bincount(list(all_train_labels))
class_weights = torch.tensor(
    [1.0/class_counts[0], 1.0/class_counts[1]],
    dtype=torch.float32
).to(DEVICE)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(dev_loader)}")
print(f"Class weights : {class_weights}")

Train batches : 195
Val batches   : 3
Class weights : tensor([0.0006, 0.0007], device='cuda:0')


**DistilBERT Model**

In [23]:
import torch.nn as nn
from transformers import DistilBertModel

class TextClassifier(nn.Module):
    def __init__(self, num_classes=2, dropout=0.3):
        super().__init__()

        self.distilbert = DistilBertModel.from_pretrained('distilbert-base-uncased')

        # Freeze bottom 4 transformer layers — fine-tune top 2 only
        for i, layer in enumerate(self.distilbert.transformer.layer):
            if i < 4:
                for param in layer.parameters():
                    param.requires_grad = False

        # Classifier head
        self.classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids, attention_mask):
        outputs    = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        # Use [CLS] token representation
        cls_output = outputs.last_hidden_state[:, 0, :]  # [B, 768]
        cls_output = self.dropout(cls_output)
        logits     = self.classifier(cls_output)          # [B, 2]
        return logits

    def get_embeddings(self, input_ids, attention_mask):
        """Returns CLS embeddings — needed for fusion later"""
        outputs    = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        cls_output = outputs.last_hidden_state[:, 0, :]
        return cls_output


text_model = TextClassifier().to(DEVICE)

# Count trainable parameters
trainable = sum(p.numel() for p in text_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in text_model.parameters())
print(f"Total parameters    : {total:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"Frozen parameters   : {total-trainable:,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Total parameters    : 66,576,322
Trainable parameters: 38,224,834
Frozen parameters   : 28,351,488


In [24]:
from sklearn.metrics import f1_score, roc_auc_score

EPOCHS    = 10   # DistilBERT needs fewer epochs than custom models
LR        = 2e-5  # standard learning rate for BERT fine-tuning
SAVE_PATH = f'{BASE_DIR}/models/text/text_distilbert_best.pt'

os.makedirs(f'{BASE_DIR}/models/text', exist_ok=True)

optimizer  = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, text_model.parameters()),
    lr=LR, weight_decay=1e-4
)
scheduler  = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=1.0, end_factor=0.1,
    total_iters=EPOCHS
)
criterion  = nn.CrossEntropyLoss(weight=class_weights)

best_f1          = 0
patience_counter = 0
patience         = 4

print("Starting training...")
print("=" * 70)

for epoch in range(EPOCHS):
    # ── Train ──
    text_model.train()
    train_loss = 0

    for batch in train_loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['label'].to(DEVICE)

        optimizer.zero_grad()
        logits = text_model(input_ids, attention_mask)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(text_model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()

    # ── Validate ──
    text_model.eval()
    val_loss, all_preds, all_labels_list, all_probs = [], [], [], []

    with torch.no_grad():
        for batch in dev_loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['label'].to(DEVICE)

            logits  = text_model(input_ids, attention_mask)
            loss    = criterion(logits, labels)
            probs   = torch.softmax(logits, dim=1)[:, 1]
            preds   = torch.argmax(logits, dim=1)

            val_loss.extend([loss.item()])
            all_preds.extend(preds.cpu().numpy())
            all_labels_list.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    avg_train = train_loss / len(train_loader)
    avg_val   = np.mean(val_loss)
    f1        = f1_score(all_labels_list, all_preds, average='macro')
    auc       = roc_auc_score(all_labels_list, all_probs)

    scheduler.step()

    if f1 > best_f1:
        best_f1          = f1
        patience_counter = 0
        torch.save(text_model.state_dict(), SAVE_PATH)
    else:
        patience_counter += 1

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Train Loss: {avg_train:.4f} | "
          f"Val Loss: {avg_val:.4f} | "
          f"F1: {f1:.4f} | "
          f"AUC: {auc:.4f} | "
          f"Best F1: {best_f1:.4f}")

    if patience_counter >= patience:
        print(f"\n⏹️ Early stopping at epoch {epoch+1}")
        break

print(f"\n✅ Training complete. Best F1: {best_f1:.4f}")

Starting training...
Epoch 01/10 | Train Loss: 0.6618 | Val Loss: 0.5983 | F1: 0.3966 | AUC: 0.4783 | Best F1: 0.3966
Epoch 02/10 | Train Loss: 0.5669 | Val Loss: 0.6958 | F1: 0.4400 | AUC: 0.4746 | Best F1: 0.4400
Epoch 03/10 | Train Loss: 0.4480 | Val Loss: 0.7090 | F1: 0.4496 | AUC: 0.5181 | Best F1: 0.4496
Epoch 04/10 | Train Loss: 0.3721 | Val Loss: 0.9492 | F1: 0.5127 | AUC: 0.5326 | Best F1: 0.5127
Epoch 05/10 | Train Loss: 0.3141 | Val Loss: 0.8118 | F1: 0.5238 | AUC: 0.5471 | Best F1: 0.5238
Epoch 06/10 | Train Loss: 0.2667 | Val Loss: 0.8549 | F1: 0.5717 | AUC: 0.5580 | Best F1: 0.5717
Epoch 07/10 | Train Loss: 0.2248 | Val Loss: 0.8475 | F1: 0.5562 | AUC: 0.5942 | Best F1: 0.5717
Epoch 08/10 | Train Loss: 0.1941 | Val Loss: 0.9066 | F1: 0.5143 | AUC: 0.6051 | Best F1: 0.5717
Epoch 09/10 | Train Loss: 0.1721 | Val Loss: 0.9957 | F1: 0.5717 | AUC: 0.6087 | Best F1: 0.5717
Epoch 10/10 | Train Loss: 0.1540 | Val Loss: 0.9741 | F1: 0.5717 | AUC: 0.6232 | Best F1: 0.5717

⏹️ Early

Overfitting, start fresh, we'll use clean daic woz dataset and train seperately and then retrain the model with tweet dataset seperately

**A.  Minimal Clean & DAIC Only**

In [25]:
import re

def clean_text_minimal(text):
    if not text:
        return None
    text = str(text)
    text = re.sub(r'http\S+', '', text)      # remove URLs only
    text = re.sub(r'\s+', ' ', text).strip() # normalize spaces only
    return text if len(text) > 0 else None

# Use DAIC-WoZ ONLY — no tweets
all_train_texts  = [clean_text_minimal(t) for t in train_texts]
all_train_labels = list(train_labels)
dev_texts_clean  = [clean_text_minimal(t) for t in dev_texts]
dev_labels_clean = list(dev_labels)

# Remove only truly None entries
train_data = [(t, l) for t, l in
              zip(all_train_texts, all_train_labels) if t]
dev_data   = [(t, l) for t, l in
              zip(dev_texts_clean, dev_labels_clean) if t]

all_train_texts,  all_train_labels = zip(*train_data)
dev_texts_clean,  dev_labels_clean = zip(*dev_data)
all_train_texts  = list(all_train_texts)
all_train_labels = list(all_train_labels)
dev_texts_clean  = list(dev_texts_clean)
dev_labels_clean = list(dev_labels_clean)

# Update class weights
import numpy as np
import torch
class_counts  = np.bincount(list(all_train_labels))
class_weights = torch.tensor(
    [1.0/class_counts[0], 1.0/class_counts[1]],
    dtype=torch.float32
).to(DEVICE)

print(f"Train: {len(all_train_texts)} | "
      f"Depressed: {sum(all_train_labels)} | "
      f"Not: {len(all_train_labels)-sum(all_train_labels)}")
print(f"Dev  : {len(dev_texts_clean)} | "
      f"Depressed: {sum(dev_labels_clean)} | "
      f"Not: {len(dev_labels_clean)-sum(dev_labels_clean)}")
print(f"Class weights: {class_weights}")
print("✅ Minimal clean — clinical context preserved")

Train: 107 | Depressed: 30 | Not: 77
Dev  : 35 | Depressed: 12 | Not: 23
Class weights: tensor([0.0130, 0.0333], device='cuda:0')
✅ Minimal clean — clinical context preserved


**B — Retokenize**

In [26]:
from transformers import BertTokenizer

tokenizer  = BertTokenizer.from_pretrained('bert-base-uncased')
MAX_LEN    = 512
BATCH_SIZE = 8

def tokenize_texts(texts, max_len=MAX_LEN):
    return tokenizer(
        texts,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

print("Tokenizing training data...")
train_encodings = tokenize_texts(all_train_texts)

print("Tokenizing validation data...")
dev_encodings   = tokenize_texts(dev_texts_clean)

print(f"✅ Tokenization complete")
print(f"Train shape: {train_encodings['input_ids'].shape}")
print(f"Dev shape  : {dev_encodings['input_ids'].shape}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Tokenizing training data...
Tokenizing validation data...
✅ Tokenization complete
Train shape: torch.Size([107, 512])
Dev shape  : torch.Size([35, 512])


**C — DataLoader**

In [27]:
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {
            'input_ids'     : self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'label'         : self.labels[idx]
        }

train_dataset = TextDataset(train_encodings, list(all_train_labels))
dev_dataset   = TextDataset(dev_encodings,   list(dev_labels_clean))

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
dev_loader    = DataLoader(dev_dataset,   batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches  : {len(dev_loader)}")
print("✅ DataLoaders ready")

Train batches: 14
Val batches  : 5
✅ DataLoaders ready


**D — Reinitialize Model**

In [28]:
import torch.nn as nn
from transformers import DistilBertModel

class TextClassifier(nn.Module):
    def __init__(self, num_classes=2, dropout=0.4):
        super().__init__()
        self.distilbert = DistilBertModel.from_pretrained('distilbert-base-uncased')

        # Freeze ALL layers
        for param in self.distilbert.parameters():
            param.requires_grad = False

        # Unfreeze only last 2 transformer layers
        for layer in self.distilbert.transformer.layer[-2:]:
            for param in layer.parameters():
                param.requires_grad = True

        # Simple classifier head
        self.classifier = nn.Sequential(
            nn.Linear(768, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids, attention_mask):
        outputs    = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        return self.classifier(cls_output)

    def get_embeddings(self, input_ids, attention_mask):
        outputs = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        return outputs.last_hidden_state[:, 0, :]

# Reinitialize fresh model
text_model = TextClassifier().to(DEVICE)
trainable  = sum(p.numel() for p in text_model.parameters() if p.requires_grad)
total      = sum(p.numel() for p in text_model.parameters())
print(f"Total parameters    : {total:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"Frozen parameters   : {total-trainable:,}")

Total parameters    : 66,461,570
Trainable parameters: 14,274,434
Frozen parameters   : 52,187,136


**E — Retrain**

In [29]:
from sklearn.metrics import f1_score, roc_auc_score

EPOCHS    = 20
LR        = 3e-5
SAVE_PATH = f'{BASE_DIR}/models/text/text_distilbert_best.pt'

import os
os.makedirs(f'{BASE_DIR}/models/text', exist_ok=True)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, text_model.parameters()),
    lr=LR, weight_decay=1e-3
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=3, factor=0.5
)
criterion = nn.CrossEntropyLoss(weight=class_weights)

best_f1          = 0
patience_counter = 0
patience         = 6

print("Starting training on DAIC-WoZ only...")
print("=" * 75)

for epoch in range(EPOCHS):
    # ── Train ──
    text_model.train()
    train_loss = 0
    for batch in train_loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['label'].to(DEVICE)

        optimizer.zero_grad()
        logits = text_model(input_ids, attention_mask)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(text_model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()

    # ── Validate ──
    text_model.eval()
    val_loss, all_preds, all_labels_list, all_probs = [], [], [], []
    with torch.no_grad():
        for batch in dev_loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['label'].to(DEVICE)

            logits = text_model(input_ids, attention_mask)
            loss   = criterion(logits, labels)
            probs  = torch.softmax(logits, dim=1)[:, 1]
            preds  = torch.argmax(logits, dim=1)

            val_loss.extend([loss.item()])
            all_preds.extend(preds.cpu().numpy())
            all_labels_list.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    avg_train = train_loss / len(train_loader)
    avg_val   = np.mean(val_loss)
    f1        = f1_score(all_labels_list, all_preds, average='macro')
    auc       = roc_auc_score(all_labels_list, all_probs)

    scheduler.step(avg_val)

    if f1 > best_f1:
        best_f1          = f1
        patience_counter = 0
        torch.save(text_model.state_dict(), SAVE_PATH)
    else:
        patience_counter += 1

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Train: {avg_train:.4f} | "
          f"Val: {avg_val:.4f} | "
          f"F1: {f1:.4f} | "
          f"AUC: {auc:.4f} | "
          f"Best: {best_f1:.4f} | "
          f"P: {patience_counter}/{patience}")

    if patience_counter >= patience:
        print(f"\n⏹️ Early stopping at epoch {epoch+1}")
        break

print(f"\n✅ Training complete. Best F1: {best_f1:.4f}")

Starting training on DAIC-WoZ only...
Epoch 01/20 | Train: 0.7134 | Val: 0.6914 | F1: 0.5238 | AUC: 0.5725 | Best: 0.5238 | P: 0/6
Epoch 02/20 | Train: 0.6961 | Val: 0.6893 | F1: 0.3966 | AUC: 0.5543 | Best: 0.5238 | P: 1/6
Epoch 03/20 | Train: 0.6884 | Val: 0.6867 | F1: 0.3966 | AUC: 0.6594 | Best: 0.5238 | P: 2/6
Epoch 04/20 | Train: 0.6868 | Val: 0.6857 | F1: 0.3966 | AUC: 0.6739 | Best: 0.5238 | P: 3/6
Epoch 05/20 | Train: 0.6808 | Val: 0.6863 | F1: 0.3966 | AUC: 0.6196 | Best: 0.5238 | P: 4/6
Epoch 06/20 | Train: 0.6652 | Val: 0.6851 | F1: 0.3966 | AUC: 0.6449 | Best: 0.5238 | P: 5/6
Epoch 07/20 | Train: 0.7029 | Val: 0.6831 | F1: 0.3966 | AUC: 0.6703 | Best: 0.5238 | P: 6/6

⏹️ Early stopping at epoch 7

✅ Training complete. Best F1: 0.5238
